In [ ]:
from urllib.parse import urlparse
import requests
import yaml
from pathlib import Path

root = Path.cwd()

# Constants from yml files
with open(root / "conf" / "config.yml", 'r') as f:
    config = yaml.safe_load(f)

with open(root / "conf" / "symbol_count.yml", 'r') as g:
    symbol_count = yaml.safe_load(g)

headers = config['headers']

session = requests.Session()

class URLFetch:

    def __init__(self, url, method='get', json=False, session=None,
                 headers = None, proxy = None):
        self.url = url
        self.method = method
        self.json = json

        if not session:
            self.session = requests.Session()
        else:
            self.session = session

        if headers:
            self.session.headers.update(headers)
        if proxy:
            self.update_proxy(proxy)
        else:
            self.update_proxy('')

    def set_session(self, session):
        self.session = session
        return self

    def get_session(self, session):
        self.session = session
        return self

    def __call__(self, *args, **kwargs):
        u = urlparse(self.url)
        self.session.headers.update({'Host': u.hostname})
        url = self.url%(args)
        if self.method == 'get':
            return self.session.get(url, params=kwargs, proxies = self.proxy )
        elif self.method == 'post':
            if self.json:
                return self.session.post(url, json=kwargs, proxies = self.proxy )
            else:
                return self.session.post(url, data=kwargs, proxies = self.proxy )

    def update_proxy(self, proxy):
        self.proxy = proxy
        self.session.proxies.update(self.proxy)

    def update_headers(self, headers):
        self.session.headers.update(headers)



In [ ]:
from functools import partial
from helper import nse_symbol_purify

URLFetchSession = partial(URLFetch, session=session,
                          headers=headers)

symbol_count_url = URLFetchSession(
    url='http://www1.nseindia.com/marketinfo/sym_map/symbolCount.jsp')

def get_symbol_count(symbol):

    symbol = nse_symbol_purify(symbol)

    try:
        return symbol_count[symbol]
    except:
        cnt = symbol_count_url(symbol=symbol).text.lstrip().rstrip()
        symbol_count[symbol] = cnt
        return cnt

In [ ]:
symbol_count_url(symbol='BAGFILMS').text

In [ ]:
import datetime

end = datetime.datetime.now().date()
start = end-datetime.timedelta(days=130)

params = {}
params['symbol'] = 'SBIN'
params['fromDate'] = start.strftime('%d-%m-%Y')
params['toDate'] = end.strftime('%d-%m-%Y')
params['series'] = 'EQ'

url = 'http://www1.nseindia.com/products/dynaContent/common/productsSymbolMapping.jsp'

In [ ]:
URLFetchSession(url='http://www1.nseindia.com/marketinfo/sym_map/symbolCount.jsp')

In [ ]:
equity_history_url_full = URLFetchSession(
    url=url)

equity_history_url = partial(equity_history_url_full,
                             dataType='PRICEVOLUMEDELIVERABLE',
                             segmentLink=3, dateRange="")

In [ ]:
equity_history_url

In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
from functools import partial

import requests
import yaml

# Constants from yml files
root = Path().cwd() 

with open(root / "conf" / "config.yml", 'r') as f:
    config = yaml.safe_load(f)


headers = config['headers']
# params = config['eq_history_params']

url = 'http://www1.nseindia.com/products/dynaContent/common/productsSymbolMapping.jsp'
params={'dataType': 'PRICEVOLUMEDELIVERABLE', 'segmentLink': 3, 'dateRange': '', 'symbol': 'INFY', 'series': 'EQ', 'symbolCount': '2', 'fromDate': '28-12-2021', 'toDate': '07-05-2022'}

session = requests.Session()
resp = session.get(url=url, params=params, headers=headers)
soup = BeautifulSoup(resp.text, 'lxml')
table = soup.find_all('table')
pd.read_html(str(table))[0]
